# Stage-gate playground — tilearray 0.3+

Drive the **OSM five-minute path** and the **simplified WMTS happy path** from a fresh checkout: edit knobs, `plan_array`, fetch, plot, check georef.

```bash
uv sync --dev
pip install matplotlib   # plotting only — not a core tilearray dependency
uv run jupyter lab       # open this notebook
```

Habit: `inspect/config → plan_array (optional) → create_array → compute_mosaic → plot`.


## Setup


In [ ]:
from __future__ import annotations

import time

import matplotlib.pyplot as plt
import numpy as np

import tilearray
from tilearray import (
    XYZConfig,
    compute_mosaic,
    create_array,
    fetch_summary,
    inspect_endpoint,
    plan_array,
)
from tilearray.fetch_presets import OSM_TILE_TEMPLATE, osm_fetch_defaults
from tilearray.types import CRS

print("tilearray", tilearray.__version__)


def box_around(x: float, y: float, width: float) -> tuple[float, float, float, float]:
    half = width / 2.0
    return (x - half, y - half, x + half, y + half)


def summarize(arr, *, wall_s: float | None = None) -> None:
    if wall_s is not None:
        print(f"wall_s={wall_s:.2f}")
    print(f"shape={arr.shape}  dims={arr.dims}  dtype={arr.dtype}")
    vals = np.asarray(arr.values)
    finite = np.isfinite(vals)
    if finite.any():
        print(
            f"  value range: {float(vals[finite].min())} .. {float(vals[finite].max())}"
            f"  finite_frac={float(finite.mean()):.3f}"
        )
    else:
        print("  value range: (no finite values)")
    for key in ("crs", "bbox_crs", "transform", "bbox_requested", "service_type", "palette_mode"):
        if key in arr.attrs:
            print(f"  {key}={arr.attrs[key]!r}")
    print(
        "  extent:",
        float(arr.x.min()),
        float(arr.y.min()),
        float(arr.x.max()),
        float(arr.y.max()),
    )


def show(arr, *, title: str = "") -> None:
    """Plot a mosaic with geographic extent.

    ``create_array`` defaults to float32. xarray ``plot.imshow(rgb="band")``
    treats floats as 0–1, so float RGB in 0–255 becomes white + cyan speckles.
    Normalise (or cast uint8) before drawing RGB.
    """
    fig, ax = plt.subplots(figsize=(6, 5))
    extent = [
        float(arr.x.min()),
        float(arr.x.max()),
        float(arr.y.min()),
        float(arr.y.max()),
    ]
    if "band" in arr.dims and arr.sizes.get("band", 0) >= 3:
        rgb = np.asarray(arr.transpose("y", "x", "band").values[..., :3])
        if np.issubdtype(rgb.dtype, np.floating):
            rgb = np.clip(rgb / 255.0, 0, 1)
        ax.imshow(rgb, extent=extent, origin="upper")
    else:
        data = np.asarray(arr).squeeze()
        ax.imshow(data, extent=extent, origin="upper", cmap="gray")
    ax.set_aspect("equal")
    if title:
        ax.set_title(title)
    fig.tight_layout()
    plt.show()


## Knobs — edit these


In [ ]:
# --- which path to run -------------------------------------------------
RUN_OSM = True
RUN_WMTS = True  # Kartverket topo — inspect → target_resolution (no URL template)

SHOW_PLAN = True
MAX_CONCURRENT = None  # e.g. 1; None = host defaults / preset

# --- OSM XYZ (edit me) -------------------------------------------------
LON, LAT = -0.005, 51.500  # central London
# LON, LAT = -0.1276, 51.5072  # Trafalgar-ish
WIDTH_DEG = 0.01  # ~1 km; keep small
ZOOM = 16
EXPAND_PALETTE = True  # False → single-band palette indices

# --- Kartverket WMTS (edit me) -----------------------------------------
WMTS_URL = "https://cache.kartverket.no/v1/service"  # opaque path — inspect probes
WMTS_LAYER = "topo"
WMTS_MATRIX_SET = "utm33n"  # native EPSG:25833 product
TARGET_RESOLUTION_M = 10.0  # metres / pixel in the matrix CRS
RESOLUTION_POLICY = "coarser"  # "nearest" | "coarser" | "finer"
WMTS_STYLE = "default"  # this host requires a named style
# Small Oslo lon/lat window (query CRS); mosaic stays EPSG:25833
WMTS_BBOX = (10.6, 59.85, 10.8, 59.95)
WMTS_BBOX_CRS = "EPSG:4326"


## OSM XYZ — plan, fetch, plot

Query AOI in lon/lat (`bbox_crs=`); mosaic `attrs["crs"]` is **EPSG:3857** over the tile union.


In [ ]:
if RUN_OSM:
    bbox = box_around(LON, LAT, WIDTH_DEG)
    bbox_crs = CRS.EPSG_4326

    extra = {}
    if MAX_CONCURRENT is not None:
        extra["max_concurrent_requests"] = MAX_CONCURRENT

    config = XYZConfig.from_url(
        OSM_TILE_TEMPLATE,
        zoom=ZOOM,
        **osm_fetch_defaults(),
        **extra,
    )

    print(f"OSM query bbox lon/lat={bbox}  zoom={ZOOM}")

    if SHOW_PLAN:
        plan = plan_array(config, bbox, bbox_crs=bbox_crs)
        print(plan)

    da = create_array(
        config,
        bbox,
        bbox_crs=bbox_crs,
        expand_palette=EXPAND_PALETTE,
    )
    t0 = time.perf_counter()
    arr = compute_mosaic(da)
    summarize(arr, wall_s=time.perf_counter() - t0)
    print(fetch_summary(da))

    crs_out = str(arr.attrs.get("crs", "")).upper()
    assert crs_out in {"EPSG:3857", "3857"}, (
        f"expected mosaic CRS EPSG:3857, got {arr.attrs.get('crs')!r}"
    )
    assert arr.attrs.get("bbox_crs") in {"EPSG:4326", "4326"}
    print("georef ok: mosaic CRS is EPSG:3857 (query was lon/lat)")

    show(arr, title="OSM XYZ  © OpenStreetMap contributors")
else:
    print("SKIP OSM (RUN_OSM=False)")


## WMTS Kartverket — inspect → resolution → mosaic

Happy path: **service + layer + matrix set + target resolution + AOI/`bbox_crs`**.
`inspect_endpoint` probes GetCapabilities when the URL has no `wmts` token (Kartverket ``/v1/service``). Pass `service_type="wmts"` to skip the probe.


In [ ]:
if RUN_WMTS:
    # Opaque URL — library probes WMTS GetCapabilities (or pass service_type="wmts")
    inspection = inspect_endpoint(WMTS_URL)
    print(inspection)

    config = inspection.build_config(
        layer=WMTS_LAYER,
        tile_matrix_set=WMTS_MATRIX_SET,
        target_resolution=TARGET_RESOLUTION_M,
        resolution_policy=RESOLUTION_POLICY,
        style=WMTS_STYLE,
    )
    print(
        "config:",
        config.layer,
        config.tile_matrix_set,
        "tile_matrix=",
        config.tile_matrix,
        "(chosen from target_resolution; template attached if caps had one)",
    )

    if SHOW_PLAN:
        plan = plan_array(config, WMTS_BBOX, bbox_crs=WMTS_BBOX_CRS)
        print(plan)
        print(
            "plan:",
            plan.matrix_id,
            plan.planned_pixel_size,
            plan.crs,
            plan.bbox_crs,
        )

    da = create_array(
        config,
        WMTS_BBOX,
        bbox_crs=WMTS_BBOX_CRS,
        show_progress=False,
    )
    t0 = time.perf_counter()
    arr = compute_mosaic(da)
    summarize(arr, wall_s=time.perf_counter() - t0)
    print(fetch_summary(da))

    assert "25833" in str(arr.attrs.get("crs", "")), (
        f"expected mosaic CRS EPSG:25833, got {arr.attrs.get('crs')!r}"
    )
    print("georef ok: mosaic CRS is matrix SupportedCRS (query was lon/lat)")

    show(arr, title="Kartverket topo WMTS  © Kartverket (CC BY 4.0)")
else:
    print("SKIP WMTS (RUN_WMTS=False)")


## Notes

- `bbox_crs=` is the **query** CRS. XYZ mosaics label **EPSG:3857**; metadata WMTS labels the matrix SupportedCRS (`attrs["bbox_crs"]` keeps the query).
- Default mosaic dtype is **float32**. For RGB plots, normalise `/255` or `astype("uint8")` — raw `plot.imshow(rgb="band")` on float 0–255 looks like white + cyan speckles.
- WMTS happy path does **not** need `url_template=` or a hand-picked TileMatrix id — use `target_resolution=`.
- OSM preset ceiling is 2 — do not raise past it for stage-gate. Kartverket: stay polite (small windows).
- Prefer lazy + `compute_mosaic`; `load_array` is eager sugar.
- Broader multi-layer pad: `stage_gate_0_2.ipynb`.

### What felt wrong?

- [ ] OSM five-minute path clear (colour map, not cyan dots)?
- [ ] WMTS inspect → resolution path clear (opaque URL probed)?
- [ ] Plan / `planned_pixel_size` / `fetch_summary` useful?
- [ ] Georef readable on plot / attrs?

```
(your notes)
```
